# FIREGUARD ID — Notebook 02: Data Preparation, Splitting & Leakage-Free Scaling
**Wilayah Pilot:** Provinsi Riau, Indonesia  
**Tujuan:** Menyiapkan data bersih, melakukan splitting train/val/test secara reproducible, dan memastikan feature scaling di-fit HANYA pada data training tanpa kebocoran informasi (data leakage).

## 5.1 & 5.2 Feature Selection & Engineering

In [ ]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib

# Muat dataset profil kerawanan grid 0.1°
df = pd.read_csv('../riau_vulnerability_grid_01deg.csv')
feature_cols = ['hotspot_count', 'fire_days', 'mean_frp', 'max_frp', 'high_conf_count']
meta_cols = ['grid_id', 'lon_min', 'lon_max', 'lat_min', 'lat_max', 'center_lon', 'center_lat', 'risk_level']

print(f'Total Grid Wilayah: {len(df)}')
df[feature_cols].describe()

## 5.4 Data Splitting (Train / Validation / Test)
Menggunakan rasio 70% Train, 15% Validation, dan 15% Test dengan `random_state=42` tetap.

In [ ]:
train_df, temp_df = train_test_split(df, test_size=0.30, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=42)

print(f'Train set:      {len(train_df):>4} sampel ({len(train_df)/len(df)*100:.1f}%)')
print(f'Validation set: {len(val_df):>4} sampel ({len(val_df)/len(df)*100:.1f}%)')
print(f'Test set:       {len(test_df):>4} sampel ({len(test_df)/len(df)*100:.1f}%)')

## 5.3 Data Transformation & Feature Scaling (Leakage-Free)
**Prinsip Penting:** `StandardScaler` di-fit HANYA pada data training untuk mencegah kebocoran statistik (data leakage) ke data validasi dan testing.

In [ ]:
scaler = StandardScaler()

# 1. FIT HANYA PADA DATA TRAINING
scaler.fit(train_df[feature_cols])

# 2. TRANSFORM KETIGA SET SECARA TERPISAH
X_train_scaled = scaler.transform(train_df[feature_cols])
X_val_scaled = scaler.transform(val_df[feature_cols])
X_test_scaled = scaler.transform(test_df[feature_cols])

print('Mean parameter hasil fit pada Train:', np.round(scaler.mean_, 3))
print('Scale parameter (std) hasil fit pada Train:', np.round(scaler.scale_, 3))

## 5.5 Analisis Balancing Data
Mengevaluasi distribusi kelas kerawanan.

In [ ]:
print('Distribusi Kelas pada Training Set:')
print(train_df['risk_level'].value_counts())

# Catatan metodologi: Pada spatial unsupervised clustering, oversampling sintetis (SMOTE)
# tidak dilakukan agar topologi geografis asli tidak terdistorsi.

## 5.6 Menyimpan Data Split & Model Preprocessing
Test set dikunci dan tidak disentuh hingga tahap evaluasi akhir.

In [ ]:
os.makedirs('../data/split', exist_ok=True)

# Simpan data mentah terpisah
train_df.to_csv('../data/split/train_raw.csv', index=False)
val_df.to_csv('../data/split/val_raw.csv', index=False)
test_df.to_csv('../data/split/test_raw.csv', index=False)

# Simpan transformer scaler
joblib.dump(scaler, '../data/split/scaler.joblib')
print('Seluruh dataset split dan scaler telah berhasil disimpan di data/split/!')